### Efficient spherical transformation

We aim at developing an efficient spherical transformation to communicate between the physical and spectral domain. Because we have no constraint on the magnetic field in the physical domain, we have to use some statistic available in the spectral domain (e.g. its spectrum). Thanks to advanced geodynamo simulations, we have a good estimate of these statistics, and we could penalize our magnetic field using them. Thus, it would allow telling the model how to balance the power correctly between the small-scales of the magnetic field (subgrid processes) and the flow.

As we are not interested into very large truncation degree of the spherical harmonics basis, we rely on methods already encoded into `scipy` to generate the associated Legendre polynomials. For the integration method, we follow the work by __Nathanaël Schaeffer (2013)__ [1], performing a Fourier transform over $\phi$ and a Gauss-Legendre transform over $\theta$.

[1]  https://doi.org/10.1002/ggge.20071

In [1]:
#imports
import pygeotools
import numpy
import scipy

In [2]:
# Initializing the library
pygeo = pygeotools.pygeotools()

pygeotools was initialized with `verbose=True`.


In [3]:
# Loading the data
pygeo.loadModel("100path", "pygeodyn_hdf5", "../geodynamo.hdf5")

In [4]:
# Defining the context
context = {"lmax": 30, "r": pygeo.constants["rCore"]}

# Defining the grid size (3°)
grid_size = 1

# Defining the grid
if not pygeo.isGrid(f"{grid_size}deg"):
    pygeo.addGrid(f"{grid_size}deg", grid_size, grid_size)

# Setting the grid
pygeo.setGrid(f"{grid_size}deg")

In [5]:
# Computing the magnetic field
MF = pygeo.addMeasure("100path", "MF", context)

30


In [6]:
# Retrieving the grid (in rad.)
_, (thetas, phis) = pygeo.getCurrentGrid()

In [7]:
# Truncation degree
l_trunc = 13

# Defining the degrees and modes
l = numpy.arange(0, l_trunc + 1, 1)
m = l[:]

# Defining mesh grids
l_grid, m_grid = numpy.meshgrid(l, m, indexing="ij")
θ_grid, φ_grid = numpy.meshgrid(thetas, phis, indexing="ij")

# Defining useful quantities
cosθ = numpy.cos(θ_grid)
sinθ = numpy.sin(θ_grid)
cos_mφ = numpy.cos(numpy.tensordot(m_grid, φ_grid, axes=0))
sin_mφ = numpy.sin(numpy.tensordot(m_grid, φ_grid, axes=0))

# Defining useful methods
def fact(x):
    return scipy.special.gamma(x + 1)

def dirac(x):
    return numpy.float32(x == 0)

# Computing the associated Legendre polynomials (ALP)
plm, _ = scipy.special.lpmn(l_trunc, l_trunc, cosθ)
plm = plm.swapaxes(0, 1)

# Computing Schmidt semi-normalisation
s_norm = numpy.sqrt((2 - dirac(m_grid)) * fact(l_grid - m_grid) / fact(l_grid + m_grid))

# Computing the Condon-Shortley phase
cs_phase = (-1)**m_grid

# Computing the well-normalized ALP
plm_norm = numpy.einsum('ijkl,ij->ijkl', plm, s_norm * cs_phase)

# Defining the real spherical harmonics
ylm_cos, ylm_sin = cos_mφ * plm_norm, sin_mφ * plm_norm

C:\Users\romai\AppData\Local\Temp\ipykernel_26124\2785241319.py:26: DeprecationWarning: `scipy.special.lpmn` is deprecated as of SciPy 1.15.0 and will be removed in SciPy 1.17.0. Please use `scipy.special.assoc_legendre_p_all` instead.
  plm, _ = scipy.special.lpmn(l_trunc, l_trunc, cosθ)


#### Testing the naive integration (summation over surface element)

In [8]:
# Selecting the radial magnetic field
Br = MF[0,...,0]

# Increments in θ and φ
dθ = numpy.gradient(thetas).mean()
dφ = numpy.gradient(phis).mean()

# Defining the surface element
dΩ = sinθ * dθ * dφ

# Computing the integral over all degrees and modes
coeffs_cos = numpy.einsum('ijkl,kl, kl->ij', ylm_cos, Br, dΩ)
coeffs_sin = numpy.einsum('ijkl,kl, kl->ij', ylm_sin, Br, dΩ)

In [ ]:
coeffs_sin[1, 1]

np.float64(nan)